# Truth-conflict axis vs. deception-probe transfer — Kaggle runner

Runs the full pipeline from `docs/PLAN.md` on a free Kaggle GPU (T4x2 or P100, 16GB).

**Sidebar settings before running:** attach the `mats12-sim-dissim` dataset via **Add Input**; Accelerator = **GPU T4 x2** (or P100); Internet = **On**; Persistence = **Files only**.

Kaggle auto-extracts the uploaded zip, so `/kaggle/input/.../mats12-sim-dissim/` already contains the project tree (read-only). The next cell copies it into the writable `/kaggle/working`, re-syncing the code each run so an updated dataset never gets shadowed by a persisted older copy.

**What this run tests (changed 2026-08-12).** Probe labels come from the verdict the model ACTUALLY gave, not from which prompt it received — the model ignores the lying instruction ~47% of the time, and labelling by prompt condition produced an instruction detector rather than a deception probe. Prompts are also format-normalised so the response's opening phrase no longer identifies the cell. Both changes alter the prompts, so **cached activations from earlier sessions are invalidated automatically** by a prompt hash and will recompute. See `results/LOG.md`.

**Session limits:** Kaggle GPU sessions cap around 9-12h and idle out after inactivity; activation caching is per-row and resumable, so a re-run picks up where it left off. A single primary run (~1,336 prompts) should finish comfortably in one sitting on T4/P100.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, pathlib, shutil, zipfile

# Kaggle auto-extracts dataset zips on upload, so /kaggle/input already holds the
# project tree (read-only). Copy it into the writable /kaggle/working.
#
# With Persistence = "Files only", /kaggle/working SURVIVES restarts — so the code is
# re-synced from the dataset on every run rather than skipped when the folder exists,
# otherwise an updated dataset would silently keep running the previous session's code.
# Generated dirs (data/cache/results/probes) are preserved so caching still resumes.
PROJECT = pathlib.Path('/kaggle/working/mats12_sim_dissim')
KEEP = {'data', 'cache', 'results', 'probes'}

hits = glob.glob('/kaggle/input/**/scripts/run_all.py', recursive=True)
if hits:
    source_root = pathlib.Path(hits[0]).resolve().parent.parent
else:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, "Couldn't find the project under /kaggle/input — is the dataset attached?"
    source_root = pathlib.Path('/kaggle/working/_unzipped')
    if source_root.exists():
        shutil.rmtree(source_root)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(source_root)

PROJECT.mkdir(parents=True, exist_ok=True)
for item in PROJECT.iterdir():                 # clear old code, keep generated data
    if item.name not in KEEP:
        shutil.rmtree(item) if item.is_dir() else item.unlink()
for item in source_root.iterdir():             # sync fresh code in
    if item.name in KEEP:
        continue
    dst = PROJECT / item.name
    shutil.copytree(item, dst) if item.is_dir() else shutil.copy2(item, dst)

print('project synced from', source_root)
print('code files:', sorted(p.name for p in (PROJECT / 'src').glob('*.py')))
print('preserved  :', sorted(p.name for p in PROJECT.iterdir() if p.name in KEEP) or 'none yet')

In [ ]:
%cd /kaggle/working/mats12_sim_dissim
# Kaggle GPU images ship torch/transformers/accelerate/sklearn/pandas/matplotlib/seaborn
# already; this just tops up what's missing. Quote the version specifier — an unquoted
# '>=' in a `!` shell cell is parsed by bash as output redirection, not pip syntax.
!pip install -q "bitsandbytes>=0.43" statsmodels scipy tqdm

## 1. CPU smoke test (no model download, ~10s)
Confirms the pipeline runs correctly in this environment before spending GPU time.

In [ ]:
!python scripts/run_all.py --dry-run --synthetic

## 2. Tiny real-model dry run (validates the GPU/model path, a few minutes)

In [ ]:
!python scripts/run_all.py --dry-run --model qwen2.5-7b-instruct

## 3. Full primary run
~1,336 prompts, 4-bit Qwen2.5-7B. Activation caching is per-row and resumable within this session — if a cell errors partway, just re-run it.

In [ ]:
!python scripts/run_all.py --model qwen2.5-7b-instruct --label-source behavior

### 3b. The control run — condition labels (the artifact)

Re-scores the SAME cached activations with the old cell-membership labelling. No new
GPU work; it reuses the cache. The comparison is the point: if behaviour labelling is
doing its job, condition labelling should show a markedly higher, flatter transfer
matrix (probes reading the instruction) than behaviour labelling does.

Results land in the same directory, so this cell overwrites `transfer_*`/`baselines.json`
— the snapshot below is taken first, and cell 3 can be re-run to restore the primary.

In [ ]:
import json, shutil, pathlib

rdir = PROJECT / 'results' / 'qwen2.5-7b-instruct'
snap = rdir / 'behavior_snapshot'
snap.mkdir(exist_ok=True)
for f in ('transfer_matrix.csv', 'transfer_long.csv', 'transfer_heatmap.png',
          'baselines.json', 'horse_race.csv', 'cells.csv', 'monotonicity.png'):
    if (rdir / f).exists():
        shutil.copy2(rdir / f, snap / f)
print('behaviour-labelled results snapshotted ->', snap)

!cd {PROJECT} && python -m src.transfer  --model qwen2.5-7b-instruct --label-source condition
!cd {PROJECT} && python -m src.baselines --model qwen2.5-7b-instruct --label-source condition

cond = json.loads((rdir / 'baselines.json').read_text())
beh = json.loads((snap / 'baselines.json').read_text())
print('\n{:<26}{:>12}{:>12}'.format('', 'behaviour', 'condition'))
for k in ('in_dist_diag_auroc', 'within_class_ood_auroc', 'style_shift_auroc',
          'behavioral_text_auroc', 'length_only_auroc', 'random_direction_floor'):
    print('{:<26}{:>12.3f}{:>12.3f}'.format(k, beh.get(k, float('nan')), cond.get(k, float('nan'))))

In [ ]:
from IPython.display import Image, display
import json, pandas as pd

rdir = PROJECT / 'results' / 'qwen2.5-7b-instruct'

# The headline diagnostic: how often did the model actually do what it was told?
# If dissimulation compliance is ~50%, condition labelling is measuring the prompt.
print('=== compliance / lie rate by cell group ===')
print(pd.read_csv(rdir / 'compliance.csv').to_string(index=False))

meta = json.loads((rdir / 'truth_axis_meta.json').read_text())
print('\ninstrument validity:', 'PASS' if meta['validity']['verdict'] else 'FAIL', meta['validity']['checks'])
print('held-out truth AUROC:', round(meta['truth_auroc_best'], 3),
      '| verdict parse rate:', round(meta['overall_parse_rate'], 3))
if meta.get('degenerate_cells'):
    print('DEGENERATE cells (one verdict for ~all rows):', meta['degenerate_cells'])

for fig in ('validity.png', 'transfer_heatmap.png', 'monotonicity.png', 'direction_cosines.png'):
    if (rdir / fig).exists():
        print('\n' + fig); display(Image(str(rdir / fig)))

print(json.dumps(json.loads((rdir / 'baselines.json').read_text()), indent=2))

## 4. Persist results
`/kaggle/working` only survives as long as this session is open, UNLESS you commit. Click **Save Version > Save & Run All (Commit)** in the top right when done — that snapshots everything under `/kaggle/working` (including `results/`, `cache/`, `data/`) as a downloadable output, and you can reattach that output as input to a future notebook to resume from cache instead of re-downloading/re-running from scratch.

Alternatively, zip and download just the results directly from this session:

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/results_export', 'zip', PROJECT / 'results')
from IPython.display import FileLink
FileLink('results_export.zip')

## 5. (Optional) Replication on a second model
Llama-3.1-8B is gated — add your HF token as a Kaggle Secret (Add-ons > Secrets) and log in first. Gemma-3-12b-it is open but larger; `--headline-only` drops the formal-style cells to cut compute.

In [ ]:
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# login(UserSecretsClient().get_secret("HF_TOKEN"))
# !python scripts/run_all.py --model llama-3.1-8b-instruct --headline-only